# 📝 정보 추출·NER 과제 LV2(응용): 정제 파이프라인·배치·비교

> LV1 에서 익힌 검증·집계를 **조합**합니다. 문제는 네 갈래로 묶여 있습니다.
>
> - **1. 정제 파이프라인**: 유형 검증·원문 등장·중복 제거를 한 함수로 묶기 · 단계별 제거 사유 세기
> - **2. 배치 집계**: 정제 결과를 논문별로 세기 · 유형별로 세기
> - **3. 규칙 기반과 LLM 견주기**: 같은 문장에 두 추출기 적용 · 역할 분담(서술형)
> - **4. 출력 형식 다루기**: 어긋난 응답에서 쓸 수 있는 개체만 건지기 · 유형 제약이 있을 때와 없을 때 · 예시를 붙여(few-shot) 형식 잡기

## 풀이 방법
1. 맨 위 **준비 셀**을 먼저 실행하세요.
2. 각 문제의 **답안 셀**을 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).

- 데이터: PMC 논문 발췌 6편(자가면역 질환과 항류마티스 약물). `data/lv2_entities.jsonl`·`data/lv2_papers.jsonl` 을 씁니다.
- 개체 유형은 교안·LV1 과 같은 5종입니다.

화이팅!

아래 준비 셀을 먼저 실행하세요.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-5.6-luna"


def make_model():
    """이 단원이 쓰는 챗 모델.

    이 모델은 temperature 를 받지 않습니다(0 을 넘기면 400 이 옵니다). 그래서 출력을 고정할
    손잡이가 없고, 같은 입력에도 답이 흔들립니다(자동화 파이프라인 단원 6절이 그 흔들림을
    여러 번 돌려 잽니다).
    """
    return ChatOpenAI(model=MODEL_NAME)


print("모델:", MODEL_NAME)

In [ ]:
# [제공 코드] 데이터 파일 읽기: 이 셀은 실행만 하세요.
import json
from pathlib import Path

# 교안은 단원 폴더에서, 정답 노트북은 정답/ 폴더에서 돌아가므로 두 경로를 모두 본다.
_DATA = Path("data") if Path("data").exists() else Path("../data")


def load_jsonl(name):
    """data/<name> 을 한 줄씩 읽어 dict 리스트로 돌려준다(한 줄에 JSON 하나)."""
    rows = []
    for line in (_DATA / name).read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def load_json(name):
    """data/<name> 을 통째로 읽어 dict 로 돌려준다(사전 파일용)."""
    return json.loads((_DATA / name).read_text(encoding="utf-8"))


print("데이터 폴더:", _DATA)

## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 개체 추출 결과(결함 포함)와 논문 원문을 훑어봅니다.

In [ ]:
# [제공 코드] 이 셀은 실행만 하세요. 문제의 재료가 되는 개체와 원문을 읽어 둡니다.
lv2_entities = load_jsonl('lv2_entities.jsonl')
lv2_papers = load_jsonl('lv2_papers.jsonl')
doc_text = {paper['doc_id']: paper['text'] for paper in lv2_papers}
allowed_types = {'Compound', 'Gene', 'Disease', 'Symptom', 'PharmacologicClass'}

print('개체 수:', len(lv2_entities), '/ 논문 수:', len(lv2_papers))
for r in lv2_entities[:5]:
    print(r)

---
# 1. 정제 파이프라인

LV1 에서 하나씩 익힌 세 검사를 **한 함수**로 묶고, 무엇이 어느 단계에서 빠졌는지까지 셉니다(교안_02 3-1~3-3).

## 1-1. 정제 파이프라인 함수 만들기
**배경**: LV1 에서 하나씩 익힌 세 가지 정제(유형 검증·원문 등장 확인·중복 제거)를 **한 함수로** 묶습니다.

**요구사항**:
- 함수 **`clean_entities(entities, doc_text, allowed)`** 를 만드세요. 아래 **순서**로 거른 리스트를 돌려줍니다.
  1. `type` 이 `allowed` 에 있는 개체만 남긴다(유형 검증).
  2. 그중 `name` 이 `doc_text[doc_id]` 원문에 있는 개체만 남긴다(원문 등장).
  3. `(name, type)` 중복을 제거한다.
- 각 원소는 입력과 같은 dict 모양을 유지합니다.

**예시**: `clean_entities(lv2_entities, doc_text, allowed_types)` 의 길이는 15 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- LV1 의 세 단계를 순서대로 이어 하나의 함수 안에서 처리한다.

세부구현:
1. 먼저 type 이 allowed 에 있는 것만 남긴다.
2. 그 결과에서 name 이 원문(doc_text[doc_id])에 있는 것만 남긴다.
3. (name, type) 을 본 집합으로 기록하며 처음 것만 모아 돌려준다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
clean = clean_entities(lv2_entities, doc_text, allowed_types)
assert len(clean) == 15, 'lv2_entities 19개에서 유형오류 1 ,  환각 2 ,  중복 1 을 빼면 15개입니다'
assert all(e in lv2_entities for e in clean), '원본 개체의 name, type, doc_id 를 그대로 유지해야 합니다'
assert all(e['type'] in allowed_types for e in clean), '허용되지 않은 유형이 남아 있습니다'
assert all(e['name'] in doc_text[e['doc_id']] for e in clean), '그 개체의 doc_id 원문과 대조했는지 확인하세요'
keys = [(e['name'], e['type']) for e in clean]
assert len(keys) == len(set(keys)), '같은 (이름, 유형) 이 두 번 남아 있습니다'
print('✅ 통과!')

In [ ]:
# [제공 코드] 1-2 부터의 입력입니다. 1-1 을 건너뛰었더라도 여기서부터 이어 풀 수 있게 다시 만듭니다.
# 1-1 을 풀었다면 여러분이 만든 clean_entities 가 그대로 쓰이고, 이 셀은 clean 만 채웁니다.
if 'clean_entities' not in globals():   # 1-1 을 풀었으면 여러분 함수를 그대로 둔다
    def clean_entities(entities, doc_text, allowed):
        """유형 검증 -> 원문 등장 확인 -> 중복 제거 순으로 거른 리스트를 돌려준다."""
        typed = [e for e in entities if e['type'] in allowed]
        present = [e for e in typed if e['name'] in doc_text[e['doc_id']]]
        seen, out = set(), []
        for e in present:
            key = (e['name'], e['type'])
            if key not in seen:
                seen.add(key)
                out.append(e)
        return out

clean = clean_entities(lv2_entities, doc_text, allowed_types)
print('정제 결과:', len(clean), '개')

## 1-2. 제거 사유 분류하기
**배경**: 정제 과정에서 **무엇이 왜** 빠졌는지 알면 데이터 품질을 진단할 수 있습니다. 단계별 제거 개수를 셉니다.

**요구사항**:
- `lv2_entities` 를 1-1 과 같은 순서로 거르며 중간 결과를 **`typed`**(유형 검증 통과)·**`present`**(원문 등장 통과) 로 남기세요.
- 각 단계에서 제거된 개수를 세어 딕셔너리 **`reasons`** 를 만드세요.
  - `'유형오류'`: 유형 검증에서 빠진 수
  - `'환각'`: 원문 등장 확인에서 빠진 수
  - `'중복'`: 중복 제거에서 빠진 수

**예시**: `reasons` 의 키는 `'유형오류'`·`'환각'`·`'중복'` 세 개이고, 값은 각 단계에서 빠진 개수입니다(세 값의 합은 19 - 15 = 4).

<details><summary>힌트</summary>

```text
접근방법:
- 각 단계의 리스트 길이 차이가 그 단계에서 제거된 수다.

세부구현:
1. 유형이 allowed_types 에 있는 것만 모아 typed 를 만든다.
2. typed 중 이름이 그 논문 원문에 있는 것만 모아 present 를 만든다.
3. 원본과 typed 의 길이 차 = 유형오류 수, typed 와 present 의 길이 차 = 환각 수.
4. present 와 1-1 clean_entities 결과의 길이 차 = 중복 수.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert all(e in lv2_entities for e in typed) and all(e['type'] in allowed_types for e in typed), \
    'typed 에는 원본 개체 중 허용 유형인 것만 담으세요'
assert len(typed) == 18, '허용 5종에 없는 유형이 하나 있습니다'
assert all(e in typed for e in present), 'present 는 typed 에서 다시 거른 결과여야 합니다'
assert len(present) == 16, '원문에 없는 개체 두 개를 더 빼야 합니다'
assert set(reasons) == {'유형오류', '환각', '중복'}, '키는 유형오류, 환각, 중복 세 개입니다'
assert sum(reasons.values()) == len(lv2_entities) - len(clean), '세 값의 합은 전체에서 정제 결과를 뺀 수입니다'
assert reasons['유형오류'] == len(lv2_entities) - len(typed), '유형 검증에서 빠진 수를 다시 세어 보세요'
assert reasons['환각'] == len(typed) - len(present), '원문 등장 확인에서 빠진 수를 다시 세어 보세요'
print('✅ 통과!')

---
# 2. 배치 집계

정제한 결과를 두 축으로 셉니다. 논문별로 한 번, 유형별로 한 번입니다(교안_02 4-1).

## 2-1. 논문별 개체 수 집계하기
**배경**: 문서 더미 전체를 정제한 뒤, **어느 논문에서 몇 개**의 개체가 나왔는지 봅니다.

**요구사항**:
- 1-1 의 `clean_entities` 로 정제한 개체를 `doc_id` 별로 세어 딕셔너리 **`per_doc`** 를 만드세요.
  (키=doc_id, 값=그 논문의 정제된 개체 수)

**예시**: `per_doc['PMC13491035']` 는 3 입니다. 논문마다 개수가 같지는 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 정제된 개체의 doc_id 를 Counter 로 센다.

세부구현:
1. clean_entities 로 정제 리스트를 만든다.
2. 각 개체의 doc_id 를 Counter 로 센다(dict 로 바꿔도 좋다).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
from collections import Counter as _Counter   # 자가채점 전용. 학생이 dict 루프로 풀었어도 이 셀이 돌게 한다

assert set(per_doc) == {paper['doc_id'] for paper in lv2_papers}, '논문 6편이 모두 키로 있어야 합니다'
assert sum(per_doc.values()) == len(clean), '합이 정제 결과 개수와 같아야 합니다(원본을 세지 않았는지 확인하세요)'
assert per_doc == dict(_Counter(e['doc_id'] for e in clean_entities(lv2_entities, doc_text, allowed_types))), \
    '정제 후 개체를 doc_id 로 센 값과 다릅니다'
print('✅ 통과!')

## 2-2. 정제 후 유형 분포 보기
**배경**: 정제된 개체가 어떤 유형에 몰려 있는지 봅니다(2-1 은 논문별, 이번은 유형별).

**요구사항**:
- 정제된 개체의 **유형별 개수**를 세어 **`type_dist`**(dict)를 만드세요.

**예시**: `type_dist['Compound']` 는 8 입니다. 나머지 유형은 직접 세어 보세요(값의 합은 정제 결과 개수와 같습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 정제 리스트의 type 을 Counter 로 센다.

세부구현:
1. clean_entities 로 정제 리스트를 만든다.
2. 각 개체의 type 을 Counter 로 세어 dict 로 만든다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
from collections import Counter as _Counter   # 자가채점 전용. 학생이 dict 루프로 풀었어도 이 셀이 돌게 한다

assert set(type_dist) <= allowed_types, '허용 5종 밖의 유형이 들어 있습니다'
assert type_dist['Compound'] == 8, 'Compound 개수를 다시 세어 보세요'
assert sum(type_dist.values()) == len(clean), '합이 정제 결과 개수와 같아야 합니다'
assert type_dist == dict(_Counter(e['type'] for e in clean_entities(lv2_entities, doc_text, allowed_types))), \
    '정제 후 개체를 type 으로 센 값과 다릅니다'
print('✅ 통과!')

---
# 3. 규칙 기반과 LLM 견주기

같은 문장에 두 추출기를 나란히 걸어 무엇이 갈리는지 보고, 어디에 무엇을 쓸지 자기 말로 정리합니다(교안_02 4-2).

## 3-1. 규칙 기반 vs LLM 비교
**배경**: 같은 문장에 규칙 기반과 LLM 을 나란히 적용해, 무엇이 더 많이 잡는지 봅니다.

**요구사항**:
- 아래 `sentence` 에 `rule_based_ner` 와 `extract_entities` 를 각각 적용해 **`rule_found`**·**`llm_found`** 에 담으세요.
- 규칙 기반이 잡은 이름 집합과 LLM 이 잡은 이름 집합을 각각 **`rule_names`**·**`llm_names`**(set)로 만드세요.

모델을 **1회** 부릅니다(규칙 기반은 호출이 없습니다).

**예시**: LLM 이 규칙 기반보다 **더 많이** 잡습니다. 사전에 없는 병 이름은 규칙 기반이 통째로 놓칩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 추출기를 같은 문장에 적용하고, 각 결과에서 이름만 집합으로 모은다.

세부구현:
1. rule_based_ner(sentence) 와 extract_entities(sentence) 를 각각 호출한다.
2. 규칙 결과는 dict 라 e['name'], LLM 결과는 객체라 e.name 으로 이름을 꺼낸다.
3. 각각 set 으로 만든다.
```

</details>

In [ ]:
# [제공 코드] 지난 시간에 만든 규칙 기반 NER: 비교용으로 다시 제공합니다(실행만 하세요).
import re

dictionary = load_json("name2id.json")
entries = dictionary["entries"]      # 소문자로 통일한 이름 -> {id, label, canonical, source}
genes = dictionary["genes"]          # 유전자 기호 -> id (대소문자 그대로)
# 평범한 영어 낱말과 철자가 같은 이름들. 소문자로 쓰인 자리는 개체로 보지 않는다.
brand_stopwords = set(dictionary["brand_stopwords"])

TOKEN = re.compile(r"[A-Za-z][A-Za-z0-9\-]*")   # 숫자와 하이픈을 낱말의 일부로 본다(CYP2C19, HLA-A)


def rule_based_ner(text):
    """사전에 있는 낱말을 (구간, 유형)이 붙은 개체 dict 리스트로 돌려준다."""
    found = []
    for match in TOKEN.finditer(text):
        word = match.group()
        if word in genes:                       # 유전자 기호는 대소문자가 뜻이라 그대로 맞춘다
            entity_type = "Gene"
        elif word.lower() in entries:
            if word.lower() in brand_stopwords and word.islower():
                continue                        # 상품명이 소문자로 쓰였으면 평범한 낱말이다
            entity_type = entries[word.lower()]["label"]
        else:
            continue                            # 사전에 없는 낱말은 여기서 통째로 빠진다(미등록어 한계)
        found.append({"name": word, "type": entity_type,
                      "start": match.start(), "end": match.end()})
    return found


print("규칙 기반 NER 준비 완료: rule_based_ner(원문)")

In [ ]:
# [제공 코드] LLM 개체 추출기: 교안의 구조화 출력 방식 그대로입니다(실행만 하세요).
from typing import Literal, Optional
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

class _Entity(BaseModel):
    name: str = Field(description='개체 이름(원문에 나온 그대로)')
    type: Literal['Compound', 'Gene', 'Disease', 'Symptom', 'PharmacologicClass', 'Other'] = Field(description='개체 유형')
    other_type: Optional[str] = Field(
        default=None,
        description="type 이 Other 일 때만 채운다. 그 개체가 실제로 무엇인지 영어 한 낱말로 (예: Year, Institution)")
    confidence: float = Field(
        description="이 개체와 유형이 맞다고 얼마나 확신하는지 0.0 에서 1.0 사이")

class _EntityList(BaseModel):
    entities: list[_Entity]

_EXTRACT_SYSTEM = '의학 논문 원문에서 개체를 뽑아라. 유형은 Compound, Gene, Disease, Symptom, PharmacologicClass 중에서 고르고, 다섯에 안 드는 개체는 Other 로 적는다. 개체 이름은 원문에 나온 표현을 그대로 쓴다.'

_EXTRACT_PROMPT = ChatPromptTemplate.from_messages([
    ('system', _EXTRACT_SYSTEM),
    ('user', '{text}'),
])

def extract_entities(text):
    chain = _EXTRACT_PROMPT | make_model().with_structured_output(_EntityList)
    return chain.invoke({'text': text}).entities

In [ ]:
# [제공 코드] 3-2 가 살펴볼 후보 모델과 설정 도구입니다(실행만 하세요).
# AutoConfig 는 config.json 만 받습니다. 가중치(수백 MB)는 받지 않습니다.
from transformers import AutoConfig

CANDIDATE_MODELS = ['dslim/bert-base-NER',            # 영어 일반
                    'Davlan/xlm-roberta-base-ner-hrl',  # 다국어 일반
                    'd4data/biomedical-ner-all']        # 영어 의료
print('후보 모델', len(CANDIDATE_MODELS), '개')

In [ ]:
# [제공 코드] 3-1, 4-2, 4-3 이 쓸 문장과 프롬프트입니다(실행만 하세요).
sentence = 'The main side-effects of this treatment are an increased risk of hypertension and diabetes.'


def make_free_prompt(sentence):
    """유형 목록을 주지 않는 프롬프트를 만든다."""
    # 자가채점이 모델에 넘어간 문구를 그대로 대조한다. 그래서 여기서 고정해 준다.
    return ('다음 문장에서 개체를 뽑아 유형과 함께 나열해 주세요.\n'
            f'문장: {sentence}')


print('문장, 프롬프트 준비 완료')

In [ ]:
# [제공 코드] 모델에 넘어간 요청을 기록해 둡니다: 이 셀은 실행만 하세요.
# 아래 문제들은 '모델을 실제로 부르는 것'이 요구 사항이라, 자가채점이 이 기록을 보고
# 답을 손으로 적어 넣지 않았는지 확인합니다.
from langchain_core.callbacks import BaseCallbackHandler

sent_prompts = []                    # 모델에 넘어간 요청 본문이 순서대로 쌓인다


class PromptRecorder(BaseCallbackHandler):
    """모델이 호출될 때마다 그 요청 본문을 sent_prompts 에 적어 둔다."""

    def on_chat_model_start(self, serialized, messages, **kwargs):
        # messages 는 [[메시지, 메시지, ...]] 꼴이다(한 번에 여러 대화를 보낼 수 있어서)
        for turn in messages:
            sent_prompts.append("\n".join(str(m.content) for m in turn))


recorder = PromptRecorder()


def make_model():
    # 준비 셀의 make_model 에 기록기를 달아 둔다. 이 뒤에 만드는 모델은 전부 여기를 지난다
    # 인자는 준비 셀과 똑같이 둡니다. 한쪽만 고치면 두 셀이 서로 다른 모델을 만듭니다
    return ChatOpenAI(model=MODEL_NAME, callbacks=[recorder])


print('요청 기록 준비 완료')

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert rule_names == {e['name'] for e in rule_found}, 'rule_names 는 rule_found 의 이름 집합입니다'
assert llm_names == {e.name for e in llm_found}, 'llm_names 는 llm_found 의 이름 집합입니다'
assert len(llm_names) > len(rule_names), 'LLM 이 더 많이 잡아야 합니다. 두 추출기를 같은 문장에 적용했는지 확인하세요'
assert any('diabetes' in n.lower() for n in llm_names), '사전에 없는 병 이름을 LLM 이 잡았는지 확인하세요'
assert not any('diabetes' in n.lower() for n in rule_names), '규칙 기반은 사전에 없는 이름을 잡지 못합니다'
assert any(sentence in prompt for prompt in sent_prompts),     '모델에 이 문장을 넘긴 기록이 없습니다. 개체를 손으로 적지 말고 extract_entities 를 부르세요'
print('✅ 통과!')

## 3-2. 이 모델을 우리 도메인에 쓸 수 있나
**배경**: 규칙 기반과 LLM 말고 **이미 학습된 NER 모델**을 받아 쓰는 길도 있습니다(교안_01 4절). 430MB 를 받기 전에 **그 모델이 우리 유형을 낼 수 있는지** 먼저 확인하는 것이 실무의 순서입니다. 설정 파일만 받아 확인합니다.

**요구사항**:
- 제공된 `CANDIDATE_MODELS` 의 각 이름에 대해 `AutoConfig.from_pretrained(이름)` 으로 설정을 받고, `config.id2label` 의 값에서 앞의 `B-`/`I-` 를 뗀 **유형 이름**만 모으세요(`'O'` 는 제외).
- 모델 이름을 키로, 그 **유형 이름의 집합**을 값으로 갖는 딕셔너리 **`model_types`** 를 만드세요.
- `model_types` 를 훑어, 우리 5종(`allowed_types`)을 **모두** 낼 수 있는 모델 이름만 리스트 **`usable`** 에 담으세요.

**확인 기준**: `model_types` 의 키는 3개이고 `dslim/bert-base-NER` 의 값은 `{'LOC', 'MISC', 'ORG', 'PER'}` 입니다. `usable` 은 **빈 리스트**입니다. 의료 전용 모델조차 `Gene` 과 `PharmacologicClass` 를 갖고 있지 않기 때문입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 모델 이름을 돌며 설정을 받아 유형 집합을 만든 뒤, 그 집합이 allowed_types 를 포함하는지 본다.

세부구현:
1. AutoConfig 는 준비 셀에서 이미 가져왔다.
2. config.id2label.values() 의 각 라벨에서 split('-', 1)[-1] 로 유형만 남긴다.
3. 'O' 는 유형이 아니므로 뺀다.
4. 집합끼리의 포함 관계는 <= 로 판정한다(allowed_types 가 모델 유형의 부분집합인가).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert set(model_types) == set(CANDIDATE_MODELS), 'CANDIDATE_MODELS 세 개가 모두 키로 있어야 합니다'
assert all(isinstance(v, set) for v in model_types.values()), '값은 유형 이름의 집합(set)이어야 합니다'
assert model_types['dslim/bert-base-NER'] == {'LOC', 'MISC', 'ORG', 'PER'},     "B-/I- 를 떼고 'O' 를 뺀 유형 이름만 남기세요"
assert len(model_types['d4data/biomedical-ner-all']) == 43, '의료 모델은 유형이 43종입니다'
assert usable == [],     '우리 5종을 모두 내는 모델은 없습니다. allowed_types <= 모델유형 으로 판정했는지 확인하세요'
assert not (allowed_types <= model_types['d4data/biomedical-ner-all']),     '의료 모델에도 Gene 과 PharmacologicClass 가 없습니다'
print('✅ 통과!')

## 3-3. 규칙 기반과 LLM 의 역할 분담 (서술형)
**배경**: 실무에서는 규칙 기반과 LLM 을 상황에 따라 골라 씁니다.

**요구사항**: 아래 markdown 셀에 다음을 **자신의 말로** 설명하세요. 정답 노트북의 모범 서술과 비교하세요.
- **표준 사전에 이미 다 들어 있는 이름**(등재된 약물 목록 등)을 대량으로 훑을 때는 어느 방식이 더 적합한지와 이유
- **최근 논문의 새 표현**(신약 이름·약효 분류 표현)에는 어느 방식이 더 적합한지와 이유

*(여기에 자신의 설명을 서술하세요. 사전에 다 있는 대량 텍스트 vs 새 표현이 나오는 최신 논문)*

---
# 4. 출력 형식 다루기

남이 넘겨준 LLM 출력처럼 **형식이 어긋난 답**에서 쓸 수 있는 것만 건져 내고, 이어서 프롬프트로 형식을 유도하는 수단을 재 봅니다(교안_02 4-3).

## 4-1. 어긋난 응답에서 쓸 수 있는 개체만 건지기
**배경**: 형식을 프롬프트로 **부탁**하면 지켜지지 않을 때가 있습니다. 앞에 설명 문장이 붙고, 펜스에 감싸이고, 어떤 개체는 칸이 통째로 빠져 옵니다. 통째로 버리는 대신 **건질 수 있는 것만 건져** 씁니다. 모델을 부르지 않으므로 결과가 매번 같습니다.

**요구사항**:
- 제공된 `broken_answer` 에서 **첫 `[` 부터 마지막 `]` 까지만** 잘라 `json.loads` 로 파싱해 **`rows`** 에 담으세요(설명 문장과 펜스가 함께 떨어져 나갑니다).
- `rows` 에서 **`'type'` 키가 있는** 개체만 골라 **`usable`** 에 담으세요. 칸이 빠진 개체를 그대로 두면 뒤 코드가 `KeyError` 로 넘어집니다.
- 전체 개수와 남은 개수를 함께 출력하고, 남은 목록도 출력하세요.

**예시**: 파싱하면 개체가 **4개**이고, 그중 쓸 수 있는 것은 **3개**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 문자열을 통째로 파싱하려 들지 말고, 대괄호 구간만 잘라 낸다.

세부구현:
1. find 로 첫 '[' 위치를, rfind 로 마지막 ']' 위치를 얻어 그 사이를 슬라이싱한다.
   1-1. 마지막 대괄호는 슬라이싱에 포함되도록 위치에 1 을 더한다.
2. json.loads 로 파싱해 rows 에 담는다.
3. 'type' 키가 있는 개체만 골라 usable 에 담는다(in 으로 키가 있는지 본다).
4. len(rows) 와 len(usable) 을 함께 출력하고 usable 도 출력한다.
```

</details>

In [ ]:
# [제공 코드] 4-1 이 되살릴 어긋난 응답입니다(실행만 하세요).
broken_answer = """요청하신 결과입니다.
```json
[{"name": "methotrexate", "type": "Compound"},
 {"name": "TNF"},
 {"name": "rheumatoid arthritis", "type": "Disease"},
 {"name": "adalimumab", "type": "Compound"}]
```"""

print(broken_answer)

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(rows, list) and len(rows) == 4, '설명 문장과 펜스를 걷어 내면 개체 네 개가 나옵니다'
assert all(isinstance(r, dict) for r in rows), 'json.loads 로 파싱한 dict 리스트여야 합니다'
assert isinstance(usable, list) and len(usable) == 3, 'type 칸이 없는 개체 하나를 빼면 세 개가 남습니다'
assert all('type' in r for r in usable), 'type 칸이 없는 개체가 usable 에 남아 있습니다'
assert all(r in rows for r in usable), 'rows 에서 고른 개체를 그대로 담아야 합니다(새로 만들지 마세요)'
assert usable == [r for r in rows if 'type' in r], 'rows 에서 type 칸이 있는 개체 전부를 순서대로 담아야 합니다'
print('✅ 통과!')

## 4-2. 프롬프트 실험: 유형을 알려 주지 않으면 어떻게 되나
**배경**: 3-1 의 `extract_entities` 는 system 문구로 유형을 **5종으로 못박은** 추출기입니다. 이번엔 유형을 **알려 주지 않는** 프롬프트를 직접 써서, 같은 문장에 무엇이 달라지는지 봅니다.

**요구사항**:
- 제공된 **`make_free_prompt(sentence)`** 로 프롬프트를 만들어 **`free_prompt`** 에 담으세요. 아래 문구가 그대로 들어 있습니다.

```text
다음 문장에서 개체를 뽑아 유형과 함께 나열해 주세요.
문장: <문장>
```

> 프롬프트를 직접 적지 않고 제공 함수를 쓰는 이유가 있습니다. 4-3 에서 **이 프롬프트에 예시만 덧붙여** 두 답을 견줄 것이라, 문구가 사람마다 다르면 그 비교가 성립하지 않습니다. **바뀌는 것이 하나뿐이어야** 차이의 원인을 짚을 수 있어요.

- `make_model()` 을 `invoke` 로 부르고, 답의 본문 문자열(`.text`)을 **`free_answer`** 에 담으세요.
- 제공된 `extract_entities` 로 같은 `sentence` 를 추출해 **`typed_ents`** 에 담고, 그 유형 집합을 **`out_types`**(set)로 만드세요.
- 두 결과를 출력해 **유형 이름이 어떻게 다른지** 눈으로 비교하세요.

모델을 **2회** 부릅니다. 자가채점이 넘어간 문구를 대조하니, 위 문구는 그대로 쓰세요.

**예시**: `out_types` 는 허용 5종의 부분집합입니다. 반면 `free_answer` 는 유형 이름을 한국어로 붙이고, 5종에 없는 분류(치료·의료 행위 같은)까지 곁들여 설명합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 프롬프트를 직접 적지 않는다. 제공된 make_free_prompt 가 문구를 글자까지 고정해 두었으니 그것을 부른다.

세부구현:
1. make_free_prompt(sentence) 를 불러 free_prompt 에 담는다.
2. make_model() 로 모델을 만들고 invoke 에 free_prompt 를 넘긴 뒤 text 속성을 꺼내 free_answer 에 담는다.
3. extract_entities(sentence) 결과를 typed_ents 에 담고, 각 개체의 type 을 set 으로 모아 out_types 를 만든다.
4. free_answer 와 out_types 를 나란히 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(free_prompt, str) and sentence in free_prompt, 'free_prompt 안에 sentence 를 f-string 으로 넣으세요'
assert 'Compound' not in free_prompt and 'Disease' not in free_prompt, '이 프롬프트는 유형 목록을 알려 주지 않아야 합니다'
assert isinstance(free_answer, str) and free_answer.strip(), 'invoke 결과의 .text 를 free_answer 에 담으세요'
assert len(typed_ents) >= 2, '제약 추출기로 개체를 여러 개 뽑아야 합니다'
assert out_types == {e.type for e in typed_ents}, 'out_types 는 typed_ents 의 type 집합입니다'
assert out_types <= allowed_types, '제약 추출기의 유형은 허용 5종 안에 있어야 합니다'
assert any('다음 문장에서 개체를 뽑아 유형과 함께 나열해 주세요.' in prompt for prompt in sent_prompts),     'free_prompt 를 모델에 넘긴 기록이 없습니다. 답을 손으로 적지 말고 invoke 로 받으세요'
assert any(word in free_answer.lower() for word in ('hypertension', 'diabetes')),     '문장에 있는 개체 이름이 free_answer 에 하나도 없습니다. invoke 결과의 .text 를 그대로 담으세요'
print('✅ 통과!')

## 4-3. 예시를 붙여(few-shot) 형식 잡기
**배경**: 4-2 의 답은 줄글 형식이 매번 다릅니다. 정답 **예시를 몇 개 붙여** 주면 모델이 그 형식을 따라 하기도 합니다. 이 방식을 **few-shot** 이라고 합니다.

**요구사항**:
- 아래 제공된 `FEW_SHOT_EXAMPLE` 을 4-2 의 `free_prompt` **앞에** 이어 붙여 **`few_prompt`** 를 만드세요. **사이에 아무것도 넣지 말고** 두 문자열을 그대로 이으세요(`FEW_SHOT_EXAMPLE + free_prompt`). 빈 줄 하나가 끼면 예시와 지시 사이가 벌어져 다른 프롬프트가 됩니다.
- `make_model()` 을 `invoke` 로 부르고, 답의 본문 문자열을 **`few_answer`** 에 담으세요.
- 4-2 의 `free_answer` 와 나란히 출력해 **형식이 어떻게 달라졌는지** 비교하세요.

모델을 **1회** 부릅니다.

**예시**: `few_answer` 는 예시 쪽으로 형식이 기웁니다. 잡히는 개체 **수**는 4-2 와 비슷합니다.

<details><summary>힌트</summary>

```text
접근방법:
- few-shot 프롬프트는 새 문장을 쓰는 것이 아니라, 예시 덩어리를 지시문 앞에 붙이는 것이다.

세부구현:
1. FEW_SHOT_EXAMPLE 과 free_prompt 를 순서대로 이어 few_prompt 를 만든다.
2. make_model() 로 모델을 만들고 invoke 에 few_prompt 를 넘겨 text 를 few_answer 에 담는다.
3. free_answer 와 few_answer 를 차례로 출력해 형식을 비교한다.
```

</details>

In [ ]:
# [제공 코드] 4-3 이 붙일 예시 덩어리입니다(실행만 하세요).
FEW_SHOT_EXAMPLE = '''예시)
원문: Carbidopa, used to treat Parkinson disease, was reported to activate AHR.
개체: Carbidopa(Compound), Parkinson disease(Disease), AHR(Gene)

위 예시와 같은 형식으로 답해 주세요.
'''

print('예시 준비 완료')

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert few_prompt.startswith(FEW_SHOT_EXAMPLE), 'FEW_SHOT_EXAMPLE 을 앞에 붙여야 합니다'
assert free_prompt in few_prompt, '4-2 의 free_prompt 를 그 뒤에 이어 붙이세요'
assert isinstance(few_answer, str) and few_answer.strip(), 'invoke 결과의 .text 를 few_answer 에 담으세요'
assert few_answer != free_answer, '예시를 붙였는데 답이 4-2 와 완전히 같다면 프롬프트가 제대로 안 바뀐 것입니다'
assert any('위 예시와 같은 형식으로 답해 주세요.' in prompt and '다음 문장에서 개체를 뽑아' in prompt
           for prompt in sent_prompts),     'few_prompt 를 모델에 넘긴 기록이 없습니다. 답을 손으로 적지 말고 invoke 로 받으세요'
assert any(word in few_answer.lower() for word in ('hypertension', 'diabetes')),     '문장에 있는 개체 이름이 few_answer 에 하나도 없습니다. invoke 결과의 .text 를 그대로 담으세요'
print('✅ 통과!')

---
수고했어요! LV2 에서 정제 파이프라인·제거 사유 분류·배치 집계·규칙 대 LLM 비교, 그리고 프롬프트를 직접 써서 유형 제약과 few-shot 의 효과를 확인했습니다. 이것으로 이 일차의 과제를 마칩니다.